# 특허 기술이전 수요예측 — Colab GPU 실행

`run_ipm_experiment.py`를 **무료 GPU(CUDA)** 에서 full 모드로 실행합니다. 맥북 뚜껑/잠자기 걱정 없이 클라우드에서 돌아갑니다.

**순서**: ① GPU → ② 코드 → ③ 라이브러리 → ④ 데이터(드라이브) → ⑤ 실행 → ⑥ 결과 저장

> ⚠️ 상단 **런타임 ▸ 런타임 유형 변경 ▸ T4 GPU** 설정. 무료 Colab은 세션 ~12h/유휴 끊김이 있으니 탭을 켜 두세요.


## ① GPU 확인


In [ ]:
!nvidia-smi


## ② 코드 받기


In [ ]:
!git clone https://github.com/joshlee614/Patent-Citation-Graph-Based-Technology-Transfer-Demand-Prediction-.git repo
%cd repo
!git log --oneline -3


## ③ 라이브러리 (torch는 이미 있음)


In [ ]:
!pip install -q torch_geometric statsmodels
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())


## ④ 데이터 (Google Drive)

드라이브에 **4개 파일만**: `patents.csv` · `transfers.csv` · `citings.csv` · `patent_embeddings.pt`

❌ `claims.csv`/`legalStatus.csv`(1.5~1.9GB)는 불필요. 예: `/content/drive/MyDrive/kipris_data`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, torch
DATA_DIR = '/content/drive/MyDrive/kipris_data'   # ← 본인 경로로 수정
EMB = os.path.join(DATA_DIR, 'patent_embeddings.pt')
for f in ['patents.csv','transfers.csv','citings.csv','patent_embeddings.pt']:
    p=os.path.join(DATA_DIR,f); print(('OK  ' if os.path.exists(p) else 'MISSING '), p)
print('embedding shape:', tuple(torch.load(EMB, map_location='cpu').shape))


## ⑤ 실행

**`--demand_sample`은 E19(Demand Score 휴리스틱) 한 줄에만 영향합니다.** 23개 메인 모델·진단은 항상 *전체* 테스트셋(약 22만 쿼리)으로 평가됩니다.
E19는 이 데이터에서 거의 무의미(orig==rev)하고 per-query 인용 BFS라 매우 느려서, **표본 200이 기본**입니다.
(E19를 더 정밀히 원하면 숫자를 키우되 느려집니다 — seed 무관 1회 계산: 2000≈수십 분. 메인 결과와는 무관.)

먼저 `--seeds 3`(빠른 검증), 통과하면 ⑥에서 10-seed.


In [ ]:
!python run_ipm_experiment.py --mode full --seeds 3 --device cuda \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" \
  --demand_sample 200 --artifact_dir ./ipm_artifacts_s3


## ⑥ 전체 10-seed (최종 결과)

E19를 논문에 더 단단히 넣고 싶으면 `--demand_sample 2000`으로 올려도 됩니다(메인 결과 불변, 1회 ~수십분 추가).


In [ ]:
!python run_ipm_experiment.py --mode full --device cuda \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" \
  --demand_sample 200 --artifact_dir ./ipm_artifacts


## 결과 확인 + 드라이브 저장


In [ ]:
print(open('ipm_artifacts/run_ipm_results.md').read())


In [ ]:
!cp -r ipm_artifacts /content/drive/MyDrive/kipris_data/ipm_results
print('결과를 드라이브 kipris_data/ipm_results 에 저장했습니다.')
